<a href="https://colab.research.google.com/github/Jorge-Ruiz-Troccoli/Data-Science-II/blob/main/Clase%20003/pandas_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas-SQL
###  Data Science
Profe Jorge Ruiz

In [ ]:
#Importar las librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3


In [ ]:
#Importar los datos
df = pd.read_csv("Encuesta.csv", sep = ";", encoding='latin1')
df.head()

In [ ]:
df.info()

In [ ]:


def crearbase():
    #de esta forma creo la base, si ya existe la lee
    con= sqlite3.connect('encuestas_it.db')
    #sqlite3.connect(ruta) devuelve un objeto de conexión, que a su vez es devuelto por crearbase().
    # Este objeto de conexión se puede utilizar para ejecutar consultas en una base de datos SQLite.

    return con

con=crearbase()
con

In [ ]:
def creartabla(con):
    # con cursor se puede agregar información a la base de datos
    cursor = con.cursor()
    sql = """
        CREATE TABLE ENCUESTA_1 (
        id INT AUTO_INCREMENT PRIMARY KEY,
        first_name VARCHAR(255),
        years INT,
        country VARCHAR(255),
        language_programming VARCHAR(255),
        other_technology VARCHAR(255),
        area_dedicated VARCHAR(255),
        age_of_experience FLOAT,
        salary_ars FLOAT
    );"""
    cursor.execute(sql)
    con.commit()

creartabla(con)

In [ ]:
df.columns

In [ ]:
# el camino más simple
df.columns = df.columns.str.strip()
df.columns

In [ ]:
# Guardar el DataFrame en la tabla 'mi_tabla' en la base de datos
df.to_sql('ENCUESTA_1', con, if_exists='replace', index=False)

In [ ]:
# Consulta SQL para seleccionar todos los datos de la tabla 'mi_tabla'
consulta = "SELECT * FROM ENCUESTA_1"

#consulta = "SELECT * FROM ENCUESTA_1 WHERE `Age of experience` > 0.5" acá es necesario usar backticks (`)

# Cargar los datos en un DataFrame
df = pd.read_sql_query(consulta, con)

df

In [ ]:
#Shape
df.shape

In [ ]:
df.info()

In [ ]:
df[df['Age of experience'].isnull()]

In [ ]:
df[df['Salary(ARS)'].isnull()]

In [ ]:
df["Country"]= df.Country.astype("category")
df["language Programming"]= df["language Programming"].astype("category")
df["Other Technology"]= df["Other Technology"].astype("category")
df["Area dedicated"]=df["Area dedicated"].astype("category")

In [ ]:
#Analisis estadistico basico
df.describe()

In [ ]:
df.describe(include="category")

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
ax=sns.barplot(x="Area dedicated", y='Age of experience', data=df)

# Rotar los nombres del eje x y mostrarlos en vertical
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
plt.show()

In [ ]:
ax=sns.barplot(x="Area dedicated", y='Salary(ARS)', data=df)

# Rotar los nombres del eje x y mostrarlos en vertical
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
plt.show()

In [ ]:
# Contar el número de observaciones en cada categoría de "Area dedicated"
counts = df['Area dedicated'].value_counts()

# Crear el gráfico circular (pie plot)
plt.pie(counts, labels=counts.index, autopct='%1.1f%%')

plt.show()

In [ ]:
df['Age of experience'].hist()


In [ ]:
df[ 'Salary(ARS)'].hist()

In [ ]:
df2=df.groupby(['Area dedicated'])[['Age of experience', 'Salary(ARS)']].median().reset_index()
# Convertir la mediana a enteros
df2[['Age of experience', 'Salary(ARS)']] = df2[['Age of experience', 'Salary(ARS)']].astype(int)
df2

#df.groupby('Area dedicated').agg({'Salary(ARS)': 'mean', 'Age of experience': 'median'}).reset_index()
#recomendable usar esto cuando queremos calcular diferentes estadisticos a las columnas

In [ ]:
# es una buena estrategia pero es un uso más avanzado

# Crear un diccionario de mapeo de los valores de df2
mapping_dict = df2.set_index('Area dedicated')['Salary(ARS)'].to_dict()
mapping_dict



In [ ]:
# Mapear los valores de df['Area dedicated'] a los valores correspondientes de df2['Salary(ARS)']

df['Filled Salary'] = df['Area dedicated'].map(mapping_dict)
df


In [ ]:
# Asignar los valores de otra columna a 'Salary(ARS)' donde corresponda
df['Salary(ARS)'].fillna(df['Filled Salary'], inplace=True)
df

In [ ]:
#opcion más avanzada recomendable para quienes tengan más experiencia
#df['Salary(ARS)'] = df.apply(lambda row: row['Salary(ARS)'] if not pd.isnull(row['Salary(ARS)']) else mapping_dict.get(row['Area dedicated']), axis=1)


In [ ]:
# hacemos lo mismo pero para datos de años de experiencia
mapping_dict_2 = df2.set_index('Area dedicated')['Age of experience'].to_dict()
mapping_dict_2

In [ ]:
df['Filled experience'] = df['Area dedicated'].map(mapping_dict_2)
df

In [ ]:
# Asignar los valores de otra columna a 'Salary(ARS)' donde corresponda
df['Age of experience'].fillna(df['Filled experience'], inplace=True)
df

In [ ]:
df.drop(["Filled Salary", "Filled experience"], axis=1, inplace=True)
df

In [ ]:
df['language Programming'].unique()

In [ ]:
df['language Programming'] = df['language Programming'].replace(['Java', 'java'], 'Java')
df['language Programming'] = df['language Programming'].replace(['Js','Javascript', 'Javasript','Javascrip' ], 'Javascript')
df['language Programming'].unique()


In [ ]:
df['language Programming']=df['language Programming'].fillna(df['language Programming'].mode()[0])
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df_9 = pd.concat([df] * 90000, ignore_index=True, axis=0)
df_9

In [ ]:
df_9.to_sql('ENCUESTA_1', con, if_exists='replace', index=False)

In [ ]:
df_9["Area dedicated"][df_9["Area dedicated"]=="Data Scientist"].count()